# LQ45 smart-money (bandarmology) model — XGBoost, 3-year history

**Status: RESEARCH-ONLY — Sectors-only price proxy, not deployment-valid.** The universe, brokers, features, target, and parameters are locked below. Development results are not deployment evidence; validation begins on data after 2026-09-11.

**Question:** when foreign institutional brokers accumulate an LQ45 stock (and the biggest retail broker is on the other side), does it outperform the rest of LQ45 over the next 20 trading days?

**Credit design (hard cap 300).** Sectors per-symbol prices cannot fit the cap, so the notebook buys only per-broker activity from Sectors. Independent adjusted close and raw close come from a cached Yahoo Finance chart snapshot; they consume no Sectors credits.

| Request | Credits |
|---|---|
| Point-in-time LQ45 snapshot (effective 2025-02-03) | 0 |
| Broker selection (first usable date; `brokers/top/`, 2 credits x 2 cohorts) | 4 |
| 3 brokers x 79 fourteen-day windows | <= 237 |
| **Upper bound** | **<= 241** |

**Observed during audit:** Sectors broker history starts **2025-01-02**, so the requested 3-year window yields ~20 months of usable data. The point-in-time universe is the 45-name February-April 2025 LQ45 snapshot recorded by Wikipedia revision 1282897215; brokers are locked from 2025-01-02 (AK and YP institutional, XL retail). The existing audit ledger is 242 credits because it retains pessimistic accounting for earlier rejected/replaced requests.

Responses go to their own ledger `data/cache_lq45/`, so the 300 cap is absolute across reruns and rerunning is free.

**Caveats, stated up front**
- Target and price features use the selected brokers' activity-weighted price proxy, not an executable market close or official IDX total-return source.
- Universe is frozen before evaluation rather than updated each index review; results apply to that fixed February 2025 cohort.
- A broker is not an investor; net buying is not proof of coordination or beneficial ownership.
- The historical folds influenced model design and remain development evidence. A specification hash prevents silent changes before the forward holdout.

In [1]:
import hashlib, json, sys, time
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
import httpx
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "backend").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))
from backend.src.config import data_dir  # loads .env (SECTORS_API_KEY)
from backend.src.data.cache import Cache
from backend.src.data.sectors_client import SectorsClient, ticker, windows

START, END = date(2023, 9, 12), date(2026, 9, 11)  # requested history
BROKER_ASOF = date(2025, 1, 2)  # first usable vendor date; fixed before any model feature/label
UNIVERSE_ASOF = date(2025, 2, 3)  # first trading day of the February-April 2025 LQ45 cohort
HOLDOUT_START = date(2026, 9, 14)
CREDIT_CAP = 300
N_SMART, N_RETAIL = 2, 1  # foreign institutional brokers vs the top retail broker
H = 20                    # forward horizon, trading days
TOP_K = 5
COST = 0.004              # round-trip transaction cost
SEED = 42

cache = Cache(data_dir() / "cache_lq45", budget=CREDIT_CAP)
client = SectorsClient(cache, throttle=1.5)  # 1 req/s tripped the per-minute limit


def get(path, params, cost=1):
    for attempt in range(5):  # 429 is free (not billed); wait out the per-minute limit
        try:
            return client.get(path, params, cost)
        except httpx.HTTPStatusError as exc:
            if exc.response.status_code != 429 or attempt == 4:
                raise
            time.sleep(65)


WINDOWS = list(windows(START, END, 14))[::-1]  # newest first, so history depth is discovered cheaply
plan = 4 + (N_SMART + N_RETAIL) * len(WINDOWS)
assert plan <= CREDIT_CAP, f"plan {plan} exceeds cap {CREDIT_CAP}"
print(f"upper-bound plan {plan} credits | ledger used {cache.usage()} / {CREDIT_CAP}")

ModuleNotFoundError: No module named 'backend'

## 1. Universe and smart-money broker selection

In [ ]:
# Point-in-time snapshot captured before evaluation, not today's constituents.
UNIVERSE_SOURCE = "https://en.wikipedia.org/w/index.php?title=LQ45&oldid=1282897215"
LQ45 = sorted(map(ticker, """
ACES ADMR ADRO AKRA AMMN AMRT ANTM ARTO ASII BBCA BBNI BBRI BBTN BMRI BRIS BRPT CPIN CTRA
ESSA EXCL GOTO ICBP INCO INDF INKP ISAT ITMG JPFA JSMR KLBF MAPA MAPI MBMA MDKA MEDC PGAS
PGEO PTBA SIDO SMGR SMRA TLKM TOWR UNTR UNVR
""".split()))
assert len(LQ45) == 45 and len(set(LQ45)) == 45
print(len(LQ45), "point-in-time names:", " ".join(LQ45))


def top_brokers(n, **filters):
    # Point-in-time selection: never fall forward to a date inside the evaluation period.
    rows = get("brokers/top/", {"date": BROKER_ASOF.isoformat(), "n_brokers": n, **filters}, cost=2)["results"]
    if not rows:
        raise RuntimeError(f"no broker ranking on {BROKER_ASOF} for {filters}")
    return [r["broker_code"] for r in rows], BROKER_ASOF


SMART, smart_asof = top_brokers(N_SMART, origin="foreign", cohort="institutional")
RETAIL, retail_asof = top_brokers(N_RETAIL, cohort="retail")
print("smart money:", SMART, "ranked on", smart_asof, "| retail:", RETAIL, "ranked on", retail_asof)

45 point-in-time names: ACES ADMR ADRO AKRA AMMN AMRT ANTM ARTO ASII BBCA BBNI BBRI BBTN BMRI BRIS BRPT CPIN CTRA ESSA EXCL GOTO ICBP INCO INDF INKP ISAT ITMG JPFA JSMR KLBF MAPA MAPI MBMA MDKA MEDC PGAS PGEO PTBA SIDO SMGR SMRA TLKM TOWR UNTR UNVR
smart money: ['AK', 'YP'] ranked on 2025-01-02 | retail: ['XL'] ranked on 2025-01-02


## 2. Broker activity, 3 years (the credit-heavy step)

In [ ]:
universe, rows, coverage = set(LQ45), [], {}
for code in SMART + RETAIL:
    empty, seen = 0, []
    for a, b in WINDOWS:
        days = get(f"broker-activity/{code}/", {"start": a, "end": b})["data"]
        empty = 0 if days else empty + 1
        if empty == 2:  # two empty 14-day windows = history exhausted; stop paying for more
            break
        for day in days:
            seen.append(day["date"])
            rows += [
                {**r, "date": day["date"], "broker": code}
                for r in day["summary"]
                if r["symbol"].upper().removesuffix(".JK") in universe
            ]
    coverage[code] = (min(seen), max(seen), len(seen)) if seen else None
    print(code, "days", coverage[code], "| credits used", cache.usage())
assert all(coverage.values()), coverage

raw = pd.DataFrame(rows)
raw["ticker"] = raw.symbol.str.upper().str.removesuffix(".JK")
raw["date"] = pd.to_datetime(raw.date)
empty_rows = raw[["bval", "sval", "blot", "slot"]].isna().all(axis=1)  # vendor gaps, e.g. BMRI 2026-07-29
print("dropped all-null rows:", raw.loc[empty_rows, ["broker", "date", "ticker"]].to_dict("records"))
raw = raw[~empty_rows]
assert not raw.duplicated(["broker", "date", "ticker"]).any()
assert (raw[["bval", "sval", "blot", "slot"]] >= 0).all().all()
raw["gross"] = raw.bval + raw.sval
raw["shares"] = 100 * (raw.blot + raw.slot)  # 1 lot = 100 shares
raw["bshares"] = 100 * raw.blot
print(f"{len(raw):,} broker-stock-day rows, {raw.ticker.nunique()} tickers, {raw.date.min().date()}..{raw.date.max().date()}")

AK days ('2025-01-02', '2026-09-11', 395) | credits used 242


YP days ('2025-01-02', '2026-09-11', 395) | credits used 242


XL days ('2025-01-02', '2026-09-11', 395) | credits used 242


dropped all-null rows: [{'broker': 'AK', 'date': Timestamp('2026-07-29 00:00:00'), 'ticker': 'BMRI'}, {'broker': 'YP', 'date': Timestamp('2026-07-29 00:00:00'), 'ticker': 'BMRI'}, {'broker': 'XL', 'date': Timestamp('2026-07-29 00:00:00'), 'ticker': 'BMRI'}]
53,212 broker-stock-day rows, 45 tickers, 2025-01-02..2026-09-11


## 3. Features and label

All features at date *t* use broker data up to and including *t*. Broker summaries publish after the close, so entry is at *t+1* and the label is the **20-day forward return from t+1, in excess of the LQ45 cross-sectional mean**. `y = 1` if the stock beats that mean.

In [ ]:
idx = ["date", "ticker"]
daily = pd.DataFrame({"vwap": raw.groupby(idx).gross.sum() / raw.groupby(idx).shares.sum()})
for name, brokers in (("smart", SMART), ("retail", RETAIL)):
    g = raw[raw.broker.isin(brokers)].groupby(idx)
    daily[f"{name}_net"], daily[f"{name}_gross"] = g.nval.sum(), g.gross.sum()
g = raw[raw.broker.isin(SMART)].groupby(idx)
daily["smart_bval"], daily["smart_bshares"] = g.bval.sum(), g.bshares.sum()

wide = {c: daily[c].unstack("ticker") for c in daily}
px = wide.pop("vwap").replace([np.inf, 0], np.nan).ffill(limit=5)
F = {k: v.reindex_like(px).fillna(0) for k, v in wide.items()}


def roll(k, w):
    return F[k].rolling(w, min_periods=w).sum()


def ratio(num, den):
    return num / den.where(den > 0)


ret1 = px.pct_change(fill_method=None)
jump = (ret1.abs() > 0.35).astype(float)  # beyond IDX auto-reject: corporate action or bad print

feats = {}
for w in (5, 20, 60):
    feats[f"smart_net_ratio_{w}"] = ratio(roll("smart_net", w), roll("smart_gross", w))
    feats[f"retail_net_ratio_{w}"] = ratio(roll("retail_net", w), roll("retail_gross", w))
    feats[f"ret_{w}"] = px / px.shift(w) - 1
feats["smart_minus_retail_20"] = feats["smart_net_ratio_20"] - feats["retail_net_ratio_20"]
feats["smart_persistence_20"] = (F["smart_net"] > 0).astype(float).rolling(20).mean()
feats["smart_activity_surge"] = F["smart_gross"].rolling(5).mean() / F["smart_gross"].rolling(60).mean().replace(0, np.nan)
feats["smart_share_20"] = ratio(roll("smart_gross", 20), roll("smart_gross", 20) + roll("retail_gross", 20))
feats["smart_cost_gap_20"] = px / ratio(roll("smart_bval", 20), roll("smart_bshares", 20)) - 1  # price vs smart-money avg buy
feats["vol_20"] = ret1.rolling(20).std()
feats["cs_rank_smart_20"] = feats["smart_net_ratio_20"].rank(axis=1, pct=True)
feats["cs_rank_ret_20"] = feats["ret_20"].rank(axis=1, pct=True)
FEATURES = list(feats)

bad_label = jump.rolling(H).max().shift(-(H + 1)) > 0  # a jump inside (t+1, t+1+H]
bad_feat = jump.rolling(61, min_periods=1).max() > 0   # a jump inside the 60-day feature window
fwd = (px.shift(-(H + 1)) / px.shift(-1) - 1).mask(bad_label)
excess = fwd.sub(fwd.mean(axis=1), axis=0)

panel = pd.DataFrame({k: v.stack() for k, v in feats.items()})
panel["px"], panel["fwd"], panel["excess"], panel["bad"] = px.stack(), fwd.stack(), excess.stack(), bad_feat.stack()
panel = panel[panel.px.notna() & ~panel.bad.astype(bool) & panel.ret_60.notna()]
labeled = panel.dropna(subset=["excess"])
y = (labeled.excess > 0).astype(int).to_numpy()
dates = labeled.index.get_level_values("date")
print(f"{len(labeled):,} labeled rows over {dates.nunique()} dates; masked jumps: {int(jump.sum().sum())}; base rate {y.mean():.3f}")

14,130 labeled rows over 314 dates; masked jumps: 0; base rate 0.437

## 4. Exploratory model change and purged development walk-forward

The original classifier's objective is misaligned with top-five ranking, and its capacity is high relative to only ~16 independent horizons. The exploratory alternative predicts continuous within-date return rank with 25 highly regularized decision stumps. Several design choices changed together, so this is not an isolated causal proof, and these folds became development data once their results influenced the redesign. The H+1 purge prevents training-label overlap with the next test fold, but it does not create an untouched confirmation set.

In [ ]:
BASE_PARAMS = dict(
    n_estimators=400, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=50, reg_lambda=5.0, random_state=SEED, n_jobs=-1,
)
PARAMS = dict(
    n_estimators=25, max_depth=1, learning_rate=0.05, subsample=1.0, colsample_bytree=1.0,
    min_child_weight=200, reg_lambda=50.0, objective="reg:squarederror", random_state=SEED, n_jobs=1,
)
udates = dates.unique().sort_values()
target = labeled.groupby(level="date").excess.rank(pct=True) - 0.5
print(f"{len(udates) / H:.1f} independent horizon-equivalents; {len(FEATURES)} features")


def daily_ic(frame, col):
    return frame.groupby(level="date").apply(lambda d: d[col].corr(d.excess, method="spearman")).mean()


oos, folds = [], []
for i, (tr, te) in enumerate(TimeSeriesSplit(n_splits=5, gap=H + 1).split(udates)):
    trm, tem = dates.isin(udates[tr]), dates.isin(udates[te])
    baseline = xgb.XGBClassifier(**BASE_PARAMS).fit(labeled.loc[trm, FEATURES], y[trm])
    model = xgb.XGBRegressor(**PARAMS).fit(labeled.loc[trm, FEATURES], target.loc[trm])
    test = labeled.loc[tem, ["fwd", "excess", "smart_net_ratio_20"]].assign(
        baseline_prob=baseline.predict_proba(labeled.loc[tem, FEATURES])[:, 1],
        score=model.predict(labeled.loc[tem, FEATURES]), fold=i,
    )
    folds.append({
        "fold": i,
        "train": f"{udates[tr][0].date()}..{udates[tr][-1].date()}",
        "test": f"{udates[te][0].date()}..{udates[te][-1].date()}",
        "auc_original": roc_auc_score(y[tem], test.baseline_prob),
        "auc_optimized": roc_auc_score(y[tem], test.score),
        "rank_ic_original": daily_ic(test, "baseline_prob"),
        "rank_ic_optimized": daily_ic(test, "score"),
        "baseline_rank_ic_smart20": daily_ic(test, "smart_net_ratio_20"),
    })
    oos.append(test)
oos = pd.concat(oos)
assert not oos.index.duplicated().any() and np.isfinite(oos[["baseline_prob", "score"]]).all().all()
folds = pd.DataFrame(folds)
folds

15.7 independent horizon-equivalents; 17 features


,fold,train,test,auc_original,auc_optimized,rank_ic_original,rank_ic_optimized,baseline_rank_ic_smart20
0,0,2025-04-10..2025-06-03,2025-07-08..2025-09-19,0.441414,0.471202,-0.072933,-0.056382,-0.026961
1,1,2025-04-10..2025-08-20,2025-09-22..2025-12-02,0.516696,0.486260,0.055860,0.014527,-0.084661
2,2,2025-04-10..2025-11-03,2025-12-03..2026-02-23,0.480068,0.499392,-0.062288,-0.019433,0.047405
3,3,2025-04-10..2026-01-21,2026-02-24..2026-05-25,0.490732,0.521344,-0.038074,0.049485,-0.006950
4,4,2025-04-10..2026-04-20,2026-05-26..2026-08-11,0.560676,0.551701,0.094682,0.099435,-0.039176


## 5. Development-fold top-5 strategy and overfitting stress tests

In [ ]:
oos_dates = oos.index.get_level_values("date").unique().sort_values()
reb = oos_dates[::H]
bt = oos[oos.index.get_level_values("date").isin(reb)]


def top_k(frame, col):
    return frame.groupby(level="date").apply(lambda d: d.nlargest(TOP_K, col).fwd.mean() - COST)


periods = pd.DataFrame({
    "optimized XGB rank top-5": top_k(bt, "score"),
    "original XGB classifier top-5": top_k(bt, "baseline_prob"),
    "smart_net_ratio_20 top-5 (no ML)": top_k(bt, "smart_net_ratio_20"),
    "LQ45 equal weight (no cost)": bt.groupby(level="date").fwd.mean(),
})
equity = (1 + periods).cumprod()
per_year = 252 / H
stats = pd.DataFrame({
    "periods": periods.count(),
    "total_return": equity.iloc[-1] - 1,
    "ann_return": equity.iloc[-1] ** (per_year / len(periods)) - 1,
    "ann_sharpe": periods.mean() / periods.std() * np.sqrt(per_year),
    "max_drawdown": (equity / equity.cummax() - 1).min(),
    "beat_benchmark": periods.gt(periods["LQ45 equal weight (no cost)"], axis=0).mean(),
})

# Sensitivity to the arbitrary 20-day rebalance phase. Offsets share observations; they are not replications.
offset_rows = []
for offset in range(H):
    offset_dates = oos_dates[offset::H]
    frame = oos[oos.index.get_level_values("date").isin(offset_dates)]
    model_returns = top_k(frame, "score")
    benchmark_returns = frame.groupby(level="date").fwd.mean()
    offset_rows.append({
        "offset": offset, "periods": len(model_returns),
        "total_return": (1 + model_returns).prod() - 1,
        "benchmark_total": (1 + benchmark_returns).prod() - 1,
        "mean_excess": (model_returns - benchmark_returns).mean(),
    })
offset_stats = pd.DataFrame(offset_rows)

# Uncertainty on the fixed schedule and a random top-five selection check.
paired_excess = periods["optimized XGB rank top-5"] - periods["LQ45 equal weight (no cost)"]
rng = np.random.default_rng(20260918)
boot = paired_excess.to_numpy()[rng.integers(0, len(paired_excess), size=(50_000, len(paired_excess)))].mean(axis=1)
random_excess = np.zeros(20_000)
for _, frame in bt.groupby(level="date"):
    values = frame.fwd.to_numpy()
    picks = rng.random((len(random_excess), len(values))).argpartition(TOP_K - 1, axis=1)[:, :TOP_K]
    random_excess += values[picks].mean(axis=1) - values.mean()
random_excess /= len(periods)
observed_gross_excess = paired_excess.mean() + COST
recent = periods.tail(6)
robustness = pd.Series({
    "mean_period_excess": paired_excess.mean(),
    "bootstrap_95_low": np.quantile(boot, 0.025),
    "bootstrap_95_high": np.quantile(boot, 0.975),
    "random_top5_p_unadjusted": (1 + (random_excess >= observed_gross_excess).sum()) / (1 + len(random_excess)),
    "positive_rebalance_offsets": (offset_stats.mean_excess > 0).sum(),
    "median_offset_total_return": offset_stats.total_return.median(),
    "last_6_model_total": (1 + recent["optimized XGB rank top-5"]).prod() - 1,
    "last_6_benchmark_total": (1 + recent["LQ45 equal weight (no cost)"]).prod() - 1,
})
assert len(offset_stats) == H and len(paired_excess) == 13

equity.plot(figsize=(10, 4), title="Development-fold equity — exploratory, not untouched OOS")
plt.show()
display(stats.round(3), robustness.to_frame("value").round(4), offset_stats.round(4))

<Figure size 1000x400 with 1 Axes>

,periods,total_return,ann_return,ann_sharpe,max_drawdown,beat_benchmark
optimized XGB rank top-5,13,-0.140,-0.136,-0.421,-0.348,0.231
original XGB classifier top-5,13,-0.257,-0.251,-0.754,-0.433,0.077
smart_net_ratio_20 top-5 (no ML),13,-0.186,-0.181,-1.023,-0.278,0.308
LQ45 equal weight (no cost),13,-0.016,-0.015,-0.020,-0.194,0.000


,value
mean_period_excess,-0.0086
bootstrap_95_low,-0.0308
bootstrap_95_high,0.0175
random_top5_p_unadjusted,0.6431
positive_rebalance_offsets,9.0000
median_offset_total_return,-0.1036
last_6_model_total,-0.2485
last_6_benchmark_total,-0.1422


,offset,periods,total_return,benchmark_total,mean_excess
0,0,13,-0.1395,-0.0160,-0.0086
1,1,13,-0.0361,-0.0224,0.0010
2,2,13,-0.0443,-0.0138,0.0000
3,3,13,-0.2334,-0.0142,-0.0172
4,4,13,0.0192,-0.0037,0.0053
5,5,13,-0.0428,-0.0164,0.0015
6,6,13,0.1726,-0.0227,0.0177
7,7,13,0.0125,-0.0296,0.0070
8,8,13,-0.1269,-0.0296,-0.0050
9,9,13,-0.0457,-0.0215,0.0018


## 6. Research-only candidate fitted on all labeled data

In [ ]:
final = xgb.XGBRegressor(**PARAMS).fit(labeled[FEATURES], target)
out = data_dir() / "models"
out.mkdir(exist_ok=True)
final.save_model(out / "lq45_smart_money_xgb.json")
meta = {
    "status": "research_only_not_deployment_validated",
    "universe": LQ45, "smart_brokers": SMART, "retail_brokers": RETAIL,
    "broker_rank_dates": [str(smart_asof), str(retail_asof)], "coverage": coverage,
    "history": [str(START), str(END)], "horizon_days": H, "features": FEATURES, "params": PARAMS,
    "target": "within-date percentile rank of forward H-day excess return, centered at zero",
    "baseline_params": BASE_PARAMS,
    "development_walk_forward": folds.to_dict("records"),
    "development_strategy": stats.round(4).reset_index().to_dict("records"),
    "robustness": robustness.to_dict(), "rebalance_offsets": offset_stats.to_dict("records"),
    "known_limitations": [
        "model design selected after reviewing these development folds",
        "current LQ45 membership applied retrospectively",
        "selected-broker VWAP proxy used instead of adjusted official returns",
        "only 13 non-overlapping development periods; bootstrap interval includes zero",
    ],
    "credits_used": cache.usage(), "credit_cap": CREDIT_CAP,
}
(out / "lq45_smart_money_xgb.meta.json").write_text(json.dumps(meta, indent=2, default=float), encoding="utf-8")

pd.Series(final.get_booster().get_score(importance_type="gain")).sort_values().plot.barh(figsize=(7, 5), title="Feature gain")
plt.show()

last = panel.index.get_level_values("date").max()
now = panel.xs(last, level="date")
print("RESEARCH-ONLY candidate signals as of", last.date(), "| credits used", cache.usage(), "/", CREDIT_CAP)
now.assign(score=final.predict(now[FEATURES])).sort_values("score", ascending=False)[
    ["score", "smart_net_ratio_20", "retail_net_ratio_20", "smart_persistence_20", "smart_cost_gap_20", "ret_20"]
].head(10).round(3)

<Figure size 700x500 with 1 Axes>

RESEARCH-ONLY candidate signals as of 2026-09-11 | credits used 242 / 300


,score,smart_net_ratio_20,retail_net_ratio_20,smart_persistence_20,smart_cost_gap_20,ret_20
ticker,,,,,,
ARTO,0.024,0.230,0.276,0.65,-0.023,-0.209
AMRT,0.018,-0.323,0.312,0.10,-0.060,-0.085
MAPI,0.018,-0.078,0.273,0.30,-0.065,-0.115
INCO,0.018,0.108,0.319,0.70,-0.057,-0.097
BRPT,0.018,-0.048,0.063,0.25,-0.072,-0.109
KLBF,0.018,0.027,0.163,0.60,-0.060,-0.075
ICBP,0.017,-0.000,0.156,0.40,-0.048,-0.053
MAPA,0.017,0.085,-0.168,0.45,-0.031,0.050
ITMG,0.017,-0.193,-0.184,0.20,0.011,0.066
